# WLTP Class 3 Traction Inverter → Operating Points

This notebook converts the standardised **WLTP Class 3** driving cycle into
electrical operating points for a traction inverter and summarises them as
weighted histograms suitable for semiconductor loss evaluation.

**Workflow**
1. Load the WLTP Class 3 speed profile (1 Hz, ~1800 s).
2. Apply a vehicle and motor model to derive mechanical power and DC-link current.
3. Visualise speed, mechanical power, and DC current over the drive cycle.
4. Build a 1-D histogram over DC current (10 bins) for loss-map lookup.
5. Build a 2-D histogram over (V_dc, I_dc) for full operating-point coverage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyplecs.mission_profile import (
    wltp_to_electrical,
    load_wltp,
    mission_profile_to_histogram,
    VehicleParams,
    MotorParams,
)

In [ ]:
# Load WLTP Class 3 speed profile
wltp = load_wltp(wltp_class=3)
print(f"Duration  : {wltp['time'].iloc[-1]:.0f} s")
print(f"Max speed : {wltp['speed_kmh'].max():.1f} km/h")
print(f"Avg speed : {wltp['speed_kmh'].mean():.1f} km/h")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(wltp["time"], wltp["speed_kmh"], linewidth=0.8, color="tab:blue")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Vehicle Speed (km/h)")
ax.set_title("WLTP Class 3 Speed Profile")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Convert speed profile to electrical quantities using default vehicle/motor parameters
vehicle = VehicleParams()   # default: 1800 kg EV, Cd=0.28, frontal area=2.3 m²
motor   = MotorParams()     # default: 150 kW PMSM, 400 V DC-link

elec = wltp_to_electrical(
    speed_kmh=wltp["speed_kmh"].values,
    time=wltp["time"].values,
    vehicle=vehicle,
    motor=motor,
)
print(f"Columns: {list(elec.columns)}")
elec.describe()

In [ ]:
# Plot speed, P_mech, and I_dc vs time (3 subplots)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(wltp["time"], wltp["speed_kmh"], linewidth=0.7, color="tab:blue")
axes[0].set_ylabel("Speed (km/h)")
axes[0].set_title("WLTP Class 3 — Electrical Operating Points")
axes[0].grid(True, alpha=0.4)

axes[1].plot(elec["time"], elec["P_mech"] / 1e3, linewidth=0.7, color="tab:green")
axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[1].set_ylabel("P_mech (kW)")
axes[1].grid(True, alpha=0.4)

axes[2].plot(elec["time"], elec["I_dc"], linewidth=0.7, color="tab:orange")
axes[2].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("I_dc (A)")
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# 1-D histogram over DC current (10 bins)
tbl_1d = mission_profile_to_histogram(elec, columns=["I_dc"], n_bins=10)

print(tbl_1d.summary())

fig, ax = plt.subplots(figsize=(9, 4))
tbl_1d.plot(ax=ax, kind="bar")
ax.set_title("WLTP Class 3 — DC Current Histogram (10 bins)")
plt.tight_layout()
plt.show()

In [ ]:
# 2-D histogram: 3 voltage bins × 8 current bins → heatmap
tbl_2d = mission_profile_to_histogram(elec, columns=["V_dc", "I_dc"], n_bins=[3, 8])

print(tbl_2d.summary())

fig, ax = plt.subplots(figsize=(10, 5))
tbl_2d.plot(ax=ax, kind="heatmap")
ax.set_title("WLTP Class 3 — Operating-Point Heatmap (V_dc × I_dc)")
plt.tight_layout()
plt.show()